### 03_gold_notebook
* Gold Layer — eCourts India Pipeline
* Flow: silver_enriched → 3 Gold aggregation tables \
        - `gold_summary`           : total cases by court_level, case_type, status, age_bucket \
        - `gold_age_distribution`  : pending cases by court_level, age_bucket, case_type \
        - `gold_subtype_breakdown` : pending cases by court_level, case_subtype, case_type


In [0]:
#==================================IMPORTS=========================================
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from delta.tables import DeltaTable


In [0]:
#============================== SPARK SESSION =====================================
spark = SparkSession.builder \
                .config('spark.sql.shuffle.partitions', '20') \
                .getOrCreate()

In [0]:
#==================================STORAGE PATHS===================================
silver_enriched_path    = "abfss://silver@saadlsecourtsindia.dfs.core.windows.net/cases_enriched/"
gold_summary_path       = "abfss://gold@saadlsecourtsindia.dfs.core.windows.net/gold_summary/"
gold_age_dist_path      = "abfss://gold@saadlsecourtsindia.dfs.core.windows.net/gold_age_distribution/"
gold_subtype_path       = "abfss://gold@saadlsecourtsindia.dfs.core.windows.net/gold_subtype_breakdown/"

In [0]:
#==================================READ SILVER Enriched=====================================
silver_df = spark.read.load(silver_enriched_path)

print(f"silver_enriched count : {silver_df.count()}")
silver_df.printSchema()

silver_enriched count : 1000
root
 |-- court_id: integer (nullable = true)
 |-- case_id: integer (nullable = true)
 |-- case_type: string (nullable = true)
 |-- case_subtype: string (nullable = true)
 |-- filing_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- last_modified: timestamp (nullable = true)
 |-- is_deleted: boolean (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- silver_transformed_at: timestamp (nullable = true)
 |-- court_name: string (nullable = true)
 |-- court_level: string (nullable = true)
 |-- parent_court_id: integer (nullable = true)
 |-- bench_name: string (nullable = true)
 |-- state_name: string (nullable = true)
 |-- judge_count: integer (nullable = true)
 |-- age_bucket: string (nullable = true)



### `GOLD SUMMARY`

In [0]:
#==================================GOLD SUMMARY====================================
# total cases by court_level, case_type, status, age_bucket
# Power BI will derive: older than 1 year count, percentages, KPI cards

gold_summary_df = silver_df \
    .groupBy("court_level", "case_type", "status", "age_bucket", "state_name") \
    .agg(count("case_id").alias("total_cases")) \
    .withColumn("gold_aggregated_at", current_timestamp())

print(f"gold_summary count : {gold_summary_df.count()}")
gold_summary_df.show(truncate=False)

gold_summary count : 223
+-----------+---------+--------+----------+-----------+-----------+--------------------------+
|court_level|case_type|status  |age_bucket|state_name |total_cases|gold_aggregated_at        |
+-----------+---------+--------+----------+-----------+-----------+--------------------------+
|District   |Criminal |Disposed|10+ Years |Gujarat    |2          |2026-06-28 17:12:57.261657|
|District   |Civil    |Pending |< 1 Year  |Gujarat    |1          |2026-06-28 17:12:57.261657|
|District   |Civil    |Pending |5-10 Years|Karnataka  |1          |2026-06-28 17:12:57.261657|
|District   |Criminal |Pending |10+ Years |Maharashtra|11         |2026-06-28 17:12:57.261657|
|District   |Civil    |Disposed|1-3 Years |Delhi      |1          |2026-06-28 17:12:57.261657|
|District   |Criminal |Pending |10+ Years |Delhi      |2          |2026-06-28 17:12:57.261657|
|High       |Criminal |Disposed|< 1 Year  |Maharashtra|2          |2026-06-28 17:12:57.261657|
|District   |Civil    |Pe

### `GOLD AGE DISTRIBUTION`

In [0]:
%sql
DROP TABLE IF EXISTS ecourts_gold.gold_age_distribution;
CREATE TABLE ecourts_gold.gold_age_distribution
USING DELTA
LOCATION 'abfss://gold@saadlsecourtsindia.dfs.core.windows.net/gold_age_distribution/';

In [0]:
#============================GOLD AGE DISTRIBUTION=================================
# pending cases only, by court_level, age_bucket, case_type
# Power BI will derive: % of total pending per bucket, civil/criminal split %

gold_age_dist_df = silver_df \
    .filter(col("status") == "Pending") \
    .groupBy("court_level", "age_bucket", "case_type", "state_name") \
    .agg(count("case_id").alias("pending_cases")) \
    .withColumn("gold_aggregated_at", current_timestamp()) \
    .withColumn("age_bucket_sort",
        when(col("age_bucket") == "< 1 Year",   1)
       .when(col("age_bucket") == "1-3 Years",  2)
       .when(col("age_bucket") == "3-5 Years",  3)
       .when(col("age_bucket") == "5-10 Years", 4)
       .when(col("age_bucket") == "10+ Years",  5)
    )

print(f"gold_age_distribution count : {gold_age_dist_df.count()}")
gold_age_dist_df.show(truncate=False)

gold_age_distribution count : 136
+-----------+----------+---------+-------------+-------------+--------------------------+---------------+
|court_level|age_bucket|case_type|state_name   |pending_cases|gold_aggregated_at        |age_bucket_sort|
+-----------+----------+---------+-------------+-------------+--------------------------+---------------+
|District   |10+ Years |Criminal |Uttar Pradesh|23           |2026-06-28 17:13:00.153234|5              |
|High       |< 1 Year  |Civil    |Tamil Nadu   |5            |2026-06-28 17:13:00.153234|1              |
|High       |10+ Years |Criminal |Maharashtra  |1            |2026-06-28 17:13:00.153234|5              |
|District   |5-10 Years|Civil    |Tamil Nadu   |3            |2026-06-28 17:13:00.153234|4              |
|Supreme    |5-10 Years|Criminal |India        |6            |2026-06-28 17:13:00.153234|4              |
|District   |< 1 Year  |Civil    |Maharashtra  |7            |2026-06-28 17:13:00.153234|1              |
|High       

### `GOLD SUB-TYPE DISTRIBUTION`

In [0]:
#============================GOLD SUBTYPE BREAKDOWN================================
# pending cases only, by court_level, case_subtype, case_type
# feeds High Court case type table in Power BI

gold_subtype_df = silver_df \
    .filter(col("status") == "Pending") \
    .groupBy("court_level", "case_subtype", "case_type") \
    .agg(count("case_id").alias("pending_cases")) \
    .withColumn("gold_aggregated_at", current_timestamp())

print(f"gold_subtype_breakdown count : {gold_subtype_df.count()}")
gold_subtype_df.show(truncate=False)

gold_subtype_breakdown count : 35
+-----------+----------------------------+---------+-------------+-------------------------+
|court_level|case_subtype                |case_type|pending_cases|gold_aggregated_at       |
+-----------+----------------------------+---------+-------------+-------------------------+
|District   |Drugs Case                  |Criminal |52           |2026-06-28 17:13:01.51982|
|District   |Land Dispute                |Civil    |23           |2026-06-28 17:13:01.51982|
|Supreme    |Curative Petition           |Civil    |5            |2026-06-28 17:13:01.51982|
|District   |Cheque Bounce               |Criminal |44           |2026-06-28 17:13:01.51982|
|District   |Injunction Suit             |Civil    |21           |2026-06-28 17:13:01.51982|
|High       |Criminal Appeal             |Criminal |8            |2026-06-28 17:13:01.51982|
|Supreme    |Special Leave Petition (Crl)|Criminal |4            |2026-06-28 17:13:01.51982|
|High       |Revision               

### `Write Gold`

In [0]:
#==================================WRITE GOLD======================================
# Gold is always overwrite : aggregations are fully recomputed every run from silver
# No MERGE needed — these are not row-level facts, they are aggregated results

def write_gold(df, path, table_name):
        df.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .save(path)
        print(f"{table_name} : written successfully")

write_gold(gold_summary_df,    gold_summary_path,  "gold_summary")
write_gold(gold_age_dist_df,   gold_age_dist_path, "gold_age_distribution")
write_gold(gold_subtype_df,    gold_subtype_path,  "gold_subtype_breakdown")

gold_summary : written successfully
gold_age_distribution : written successfully
gold_subtype_breakdown : written successfully


### `VALIDATION`

In [0]:
#==================================VALIDATION======================================
print("=== gold_summary by court_level ===")
gold_summary_df.groupBy("court_level").agg(sum("total_cases").alias("total")).show()

print("=== gold_age_distribution by age_bucket ===")
gold_age_dist_df.groupBy("age_bucket").agg(sum("pending_cases").alias("pending")).orderBy("age_bucket").show()

print("=== gold_subtype_breakdown by case_type ===")
gold_subtype_df.groupBy("case_type").agg(sum("pending_cases").alias("pending")).show()

=== gold_summary by court_level ===
+-----------+-----+
|court_level|total|
+-----------+-----+
|       High|  200|
|    Supreme|   50|
|   District|  750|
+-----------+-----+

=== gold_age_distribution by age_bucket ===
+----------+-------+
|age_bucket|pending|
+----------+-------+
| 1-3 Years|    187|
| 10+ Years|     96|
| 3-5 Years|    123|
|5-10 Years|    141|
|  < 1 Year|    185|
+----------+-------+

=== gold_subtype_breakdown by case_type ===
+---------+-------+
|case_type|pending|
+---------+-------+
|    Civil|    255|
| Criminal|    477|
+---------+-------+

